In [1]:
import os
import numpy as np
import importlib
import pandas as pd
import obj_2_pcd
importlib.reload(obj_2_pcd)

tiff_dir_root = '/data/jhahn/data/brain_lightsheet/slices'
obj_dir_root = '/data/jhahn/data/shape_dataset/data/brain_lightsheet_2'

dataset_annotation_file_name = obj_dir_root+"/data.csv"

#_df  = obj_2_pcd.create_dataset(dataset_annotation_file_name, tiff_dir_root, obj_dir_root)
#print(_df.head())
data_loader = obj_2_pcd.create_data_loader(dataset_annotation_file_name)
print(f"✅ DataLoader 생성 완료. 총 배치 개수: {len(data_loader)}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
data length: 28530
✅ DataLoader 생성 완료. 총 배치 개수: 28530


In [3]:
import torch
import pandas as pd
import numpy as np
import torch.multiprocessing as mp
from tqdm import tqdm
import importlib
import SlicedVolumeDataset
importlib.reload(SlicedVolumeDataset)

import obj_2_pcd
importlib.reload(obj_2_pcd)
# 데이터 변환 (예시)


if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")


### 2. DataLoader 순회 (Iteration)
tasks_to_run = []
# 일반적으로 훈련 루프(Training Loop)에서 사용됩니다.


# DataLoader를 순회하며 배치 단위로 데이터(이미지)와 레이블을 가져옵니다.
for batch_idx, (tiff_images, labels, output_dir) in enumerate(data_loader):
    
    missing_slices_list = [t.item() for t in labels['missing_slices_list']]
    _tiff_images = [t[0] for t in tiff_images]
    _output_dir = output_dir[0]


    #print((_tiff_images, labels['tickness']
    #        ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb', device))
    tasks_to_run.append((_tiff_images, labels['tickness'].item()
            ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb' ))
    
    if len(tasks_to_run) > 10:
        break
mp.set_start_method('spawn', force=True)
with mp.Pool( ) as pool: # Use a pool of 4 processes
    pool.starmap(obj_2_pcd.tiff_lilst_2_brain_obj, tqdm(tasks_to_run, total=len(tasks_to_run), desc="_tiff_2_pcd_func"))

print("DONE!")

_tiff_2_pcd_func: 100%|██████████| 11/11 [00:00<00:00, 1546.00it/s]


DONE!
